# 06 · Modelo de difusión condicional

**Taller B5-T1 · Generación de datos financieros sintéticos**

> **Notebook pendiente de asignar.** Pietro y Alonso deben repartirse entre
> ellos este notebook, el del otro generador y el de la comparativa final.
> Conviene decidirlo antes de empezar para no duplicar trabajo.
>
> Los bloques marcados como **PENDIENTE** son los que hay que completar. El
> resto, incluida la carga de datos y el guardado de resultados, ya esta
> resuelto y no conviene modificarlo: es lo que garantiza que los resultados de
> los cuatro generadores sean comparables entre si.
>
> Antes de empezar, leer `docs/GUIA_EQUIPO.md` y usar
> `04_generador_cgan.ipynb` como referencia de estructura y de estilo.

## Preparación del entorno

Se fija el backend de cómputo, se añade el código común del proyecto a la ruta
de importación y se aplica el estilo gráfico compartido. El bloque funciona sin
cambios tanto en una instalación local como en Colab.

In [ ]:
import os
import sys

# Keras 3 se ejecuta sobre PyTorch: la versión de Python empleada no dispone de
# TensorFlow y la API de capas y modelos es idéntica en ambos backends.
os.environ["KERAS_BACKEND"] = "torch"

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    !pip install -q yfinance keras torch
    # En Colab se asume que el repositorio está clonado en el directorio actual.
    RAIZ = "/content/B5-T1"
else:
    RAIZ = os.path.dirname(os.getcwd())

if os.path.join(RAIZ, "src") not in sys.path:
    sys.path.insert(0, os.path.join(RAIZ, "src"))

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from miax_b5t1 import config, datos, experimento, graficos, modelo

graficos.aplicar_estilo()
config.asegurar_directorios()

print(f"Keras {keras.__version__} sobre backend {keras.backend.backend()}")
print(f"Raíz del proyecto: {RAIZ}")

## 1. Datos de partida

Se carga el dataset y se extrae el presupuesto de datos reales fijado en el
notebook 02.

**El generador se entrena únicamente con `X_real` e `y_real`.** No debe emplearse
el conjunto de entrenamiento completo, ni validación, ni test. Si el generador
accede a datos que el clasificador no tiene, la comparación pierde validez.

In [ ]:
d = datos.cargar_dataset()

X_real, y_real = datos.submuestra_real(
    d["X_train"], d["y_train"], config.N_REALES, config.SEMILLA)

X_val, y_val = d["X_val"], d["y_val"]
X_test, y_test = d["X_test"], d["y_test"]

n_pasos, n_activos = X_real.shape[1], X_real.shape[2]
DIMENSION = n_pasos * n_activos
X_plano = X_real.reshape(len(X_real), DIMENSION)

print(f"Presupuesto real: {X_real.shape}")
print(f"Positivas: {int(y_real.sum())} ({y_real.mean():.1%})")
print(f"Dimensión de la ventana aplanada: {DIMENSION}")

## 2. Fundamento del modelo

**PENDIENTE:** redactar dos o tres parrafos explicando el modelo. Conviene cubrir:

- El proceso directo, que añade ruido gaussiano a una ventana real de forma
  progresiva hasta convertirla en ruido puro, y el hecho de que este proceso no
  tiene parámetros que aprender.
- El proceso inverso, que es lo único que se entrena: una red que, dada una
  ventana ruidosa y el instante del proceso, estima el ruido que se le anadio.
- Como se genera una muestra nueva: partiendo de ruido puro y aplicando la red de
  forma iterativa hasta recuperar una ventana limpia.
- Por que la información temporal debe entrar en la red, habitualmente mediante
  una capa de incrustación sobre el índice del paso.
- Como se incorpora el condicionamiento por la etiqueta.
- Que cabe esperar en este problema. La difusión suele reproducir mejor que otros
  modelos las distribuciones con colas pesadas, pero su muestreo es
  considerablemente más lento porque exige recorrer todos los pasos del proceso
  inverso para cada muestra.

## 3. Arquitectura

**PENDIENTE:** definir la red que estima el ruido.

Orientaciones:

- Basta una red densa que reciba la ventana aplanada ruidosa, la incrustación del
  paso temporal y la etiqueta, y devuelva un vector de la misma dimensión que la
  entrada. No hace falta una arquitectura convolucional en U.
- Un valor de partida razonable para el número de pasos del proceso esta entre
  100 y 300. Más pasos mejoran la calidad pero encarecen mucho la generación, y
  aquí se ejecuta sobre CPU.
- Conviene emplear una planificación lineal o cosenoidal de la varianza y
  precalcular los coeficientes acumulados antes del bucle de entrenamiento.
- La salida no lleva activación, ya que la red estima ruido y no una ventana.
  Tras el muestreo se recorta el resultado al intervalo `[-1, 1]`.

Aviso sobre el entorno: se emplea Keras 3 sobre PyTorch, de modo que puede
implementarse con `keras.layers` o directamente con `torch.nn`. Si se opta por
`torch.nn`, conviene mantener la estructura del notebook y el contrato de
guardado, que son independientes de la biblioteca elegida.

Al terminar, mostrar el número de parámetros de la red.

In [ ]:
from keras import layers, Model

N_PASOS_DIFUSION = 200   # ajustar si procede

# PENDIENTE: planificación de la varianza y red estimadora del ruido.

## 4. Entrenamiento

**PENDIENTE:** entrenar la red registrando la pérdida.

En cada iteración se toma un lote de ventanas reales, se sortea un paso temporal
para cada una, se les añade el ruido correspondiente a ese paso y se entrena la
red a predecir ese ruido. La pérdida habitual es el error cuadratico medio entre
el ruido real y el estimado.

Guardar el historial en una variable llamada `historial_perdida`. A diferencia de
un esquema adversario, aquí la pérdida debe descender de forma clara, de modo que
la curva es fácil de interpretar.

In [ ]:
# PENDIENTE: bucle de entrenamiento.
# historial_perdida = ...

## 5. Curva de convergencia y generación

Tras comprobar la convergencia se generan las muestras recorriendo el proceso
inverso. Conviene generar por lotes: recorrer todos los pasos del proceso para
cada muestra es la parte más costosa del notebook.

In [ ]:
# PENDIENTE: representar la curva de pérdida.

In [ ]:
N_SINTETICOS = int(config.N_REALES * max(config.RATIOS_SINTETICOS))

# PENDIENTE: muestreo del proceso inverso, condicionado por la etiqueta.
#
# X_synth = ...
# y_synth = ...

## 6. Validación cualitativa

Antes de medir el efecto sobre el clasificador conviene comprobar si las muestras
generadas se parecen a ventanas de mercado reales. El análisis exploratorio
identifico tres propiedades que un generador debería reproducir: colas más
pesadas que las de una normal, agrupamiento de la volatilidad dentro de la
ventana y correlación positiva entre activos.

Conviene comprobar también que las trayectorias no sean planas ni esten saturadas
en los extremos del intervalo, dos síntomas habituales de un generador que no ha
convergido.

In [ ]:
graficos.comparar_trayectorias(
    X_real, X_synth, "Ventanas reales frente a sintéticas", "06_trayectorias")
plt.show()

In [ ]:
graficos.comparar_distribuciones(
    X_real, X_synth, "Propiedades estadísticas de las muestras generadas",
    "06_distribuciones")
plt.show()

La última comprobación mide cuanta novedad aportan las muestras. Un generador que
se límite a reproducir el conjunto de entrenamiento producira muestras muy
próximas a las reales, y en ese caso no puede aportar información que el
clasificador no tuviera ya.

In [ ]:
fig, ax, resumen_novedad = graficos.novedad_vecino_mas_cercano(
    X_real, X_synth, "Novedad de las muestras generadas", "06_novedad")
plt.show()

for clave, valor in resumen_novedad.items():
    print(f"{clave}: {valor:.4f}")

**PENDIENTE:** comentar aquí que reproduce bien el generador y que no, apoyándose
en las tres figuras anteriores. Interesa el detalle concreto: si la correlación
entre activos se conserva, si las colas son comparables a las reales, si las
muestras aportan configuraciones nuevas o son variantes de las existentes.

## 7. Evaluación

Se ejecuta el protocolo común de evaluación. La función `barrido_ratios` entrena
el clasificador con el presupuesto real más cantidades crecientes de datos
sintéticos y repite cada configuración con varias semillas.

No conviene sustituir esta llamada por un bucle propio: es lo que asegura que los
resultados de los cuatro generadores se hayan obtenido en condiciones idénticas y
puedan compararse en el notebook final.

In [ ]:
resultados, historias = experimento.barrido_ratios(
    "diffusion", X_real, y_real, X_synth, y_synth, X_val, y_val, X_test, y_test)

experimento.guardar_sinteticos("diffusion", X_synth, y_synth,
                               perdidas={"total": historial_perdida})
ruta = experimento.guardar_resultados(resultados, "diffusion")
print(f"\nResultados guardados en {ruta}")

resumen = experimento.resumir(resultados)
display(resumen.round(4))

In [ ]:
graficos.curva_ratios(
    resumen, titulo="Efecto del generador sobre el test",
    nombre_fichero="diffusion_curva_ratios")
plt.show()

base = resumen.loc[resumen["ratio"] == 0.0, "pr_auc_media"].iloc[0]
mejor = resumen.loc[resumen["pr_auc_media"].idxmax()]
print(f"PR-AUC sin sintéticos: {base:.4f}")
print(f"Mejor PR-AUC:          {mejor['pr_auc_media']:.4f} con ratio {mejor['ratio']:g}")

Las curvas de pérdida de cada entrenamiento del clasificador documentan la
convergencia exigida por el enunciado.

In [ ]:
ratios_mostrados = sorted({r for r, _ in historias})
fig, axes = plt.subplots(1, len(ratios_mostrados),
                         figsize=(3.0 * len(ratios_mostrados), 3.2), sharey=True)
axes = np.atleast_1d(axes)

for ax, ratio in zip(axes, ratios_mostrados):
    for (r, semilla), historia in historias.items():
        if r == ratio:
            ax.plot(historia["val_loss"], linewidth=1.2, alpha=0.85)
    ax.set_title(f"ratio {ratio:g}")
    ax.set_xlabel("epoch")

axes[0].set_ylabel("pérdida en validación")
fig.suptitle("Convergencia del clasificador por cantidad de sintéticos",
             fontweight="bold")
fig.tight_layout()
graficos.guardar(fig, "diffusion_perdidas_clasificador")
plt.show()

## Conclusiones del notebook

**PENDIENTE:** cuatro o cinco puntos que respondan a:

1. Que propiedades de los datos reproduce el generador y cuales no, con atención
   particular a las colas de la distribución.
2. Si las muestras aportan configuraciones nuevas o son variantes de las
   existentes.
3. Que efecto tiene sobre el clasificador al aumentar la proporción de
   sintéticos, en validación y en test.
4. Como se compara con el generador por ruido del notebook 03.
5. Que coste computacional tiene frente a los demás generadores, dato relevante
   para valorar si compensa.

Los resultados desfavorables se documentan igual que los favorables.